## Exposure Time Calculator for SDSU MLO Spectrograph

In this notebook, we will develop a rudimentary exposure time calculator that either a) given an exposure time, outputs the SNR or b) given a SNR, outputs the exposure time.

In [23]:
#import libs:
import numpy as np
import astropy.units as u
from astropy.constants import h, c

### Basic setup:

Use equation: $$m - m_{zp} = -2.5 \log_{10}([\mathrm{counts}])$$ to get counts.

For SNR: $$SNR = \frac{S_{obj}}{\sqrt{S_{obj} + S_{sky} + S_{dark} + R^2}}$$

This translates to the CCD equation: $$SNR = \frac{S_{obj}\sqrt{Q_e t} }{\sqrt{S_{obj} + n_{pix}(S_{sky} + (\frac{S_{dark}}{Q_e}) + (\frac{R^2}{Q_e t}))}}$$

$Q_e$: quantum efficiency

$S_{obj}$: object counts (phot/sec)

$S_{sky}$: sky counts (background??) (phot/sec/pix)

$S_{dark}$: dark current (elec/sec/pix)

$n_{pix}$: pixel size of object

$R$: readout noise (elec/pix)

$t$: **exposure time**

*probably use algebra to reverse this equation?*

In [2]:
def get_counts_from_mag(mag, mag_zp):
    """Given an apparent magnitude and zero-point magnitude, output the number of counts asscociated with the magnitude."""
    counts = np.power(10, (mag_zp - mag)/2.5)

    return counts

def basic_get_SNR(exp_time, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise):
    """Predicts the SNR given an exposure time.
    
    Parameters:

    exp_time: exposure time (s)

    mag: apparent magnitude of object

    mag_zp: zero-point magnitude

    ob_size: size of object, in pixels

    sky_noise: background counts per pixel (phot/s/pix)

    dark_current: dark current (elec/s/pix)

    quant_eff: quantum efficiency (between 0 and 1)

    read_noise: time independent detector noise (elec/pix)
    """
    s_ob = get_counts_from_mag(mag, mag_zp)

    numerator = s_ob * np.sqrt(quant_eff * exp_time)
    
    denominator = np.sqrt(s_ob + ob_size * (sky_noise + (dark_current/quant_eff) + (read_noise**2/(quant_eff * exp_time))))

    snr = numerator/denominator
    
    return snr

In [3]:
def basic_get_exptime(SNR, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise):
    """Predicts the best exposure time to achieve a desired SNR.
    
    Parameters:

    SNR: signal to noise ratio

    mag: apparent magnitude of object

    mag_zp: zero-point magnitude

    ob_size: size of object, in pixels

    sky_noise: background counts per pixel (phot/s/pix)

    dark_current: dark current (elec/s/pix)

    quant_eff: quantum efficiency (between 0 and 1)

    read_noise: time independent detector noise (elec/pix)
    """

    s_ob = get_counts_from_mag(mag, mag_zp)

    C = s_ob/ob_size + sky_noise + dark_current/quant_eff

    exp_time = (C * ob_size * SNR**2 * quant_eff + np.sqrt((SNR**2 * quant_eff * ob_size * C)**2 + 4 * (s_ob * quant_eff)**2 * (read_noise**2 * ob_size * SNR**2)))/(2 * (s_ob * quant_eff)**2)

    return exp_time

In [4]:
#create some mock values to test:

mag = 16
mag_zp = 24
exp_time = 30.0

pix_rad = 5
ob_size = np.pi * pix_rad**2

dark_current = 0.01
sky_noise = 5.0
quant_eff = 0.75
read_noise = 5.0

basic_get_SNR(exp_time, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise)

np.float64(165.4001836679808)

In [5]:
des_SNR = 100
basic_get_exptime(des_SNR, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise)

np.float64(11.691419511902296)

### Better approach: Calculate $S_{ob}$ NOT from zero point:

Use the equation: $$S_{obj}​=∫f_λ​(λ)\frac{λ}{hc}​A_{tel}​T_{total}​(λ)dλ$$

where $f_{\lambda}$ is the flux density. To transform a magnitude into this quantity, use: $$f_{\nu}​=3631\mathrm{Jy}×10^{−m/2.5}$$ then 
$$f_{\lambda} = f_{\nu}\frac{c}{\lambda^2}$$

$T_{total}$ is the total throughput, which is defined as: $$T_{total}​=T_{fiber}​T_{grating}​T_{lenses}​T_{detector}​​$$

$A_{tel}$ is the telescope area.

**Must get rid of Quantum efficiency in CCD equation if using this approach**

In [64]:
def get_tp(fiber=0.995, grat=0.6, lens=0.99, detec=0.8):
    """Obtains the total instrument throughput from individual components"""

    total_tp = fiber * grat * lens * detec
    print('Throughput parameters:\n------------------------')
    print(f'Fiber: {fiber}\nLenses: {lens}\nGrating: {grat}\nDetector: {detec}\nTotal Throughput: {total_tp:.3f}')

    return total_tp

In [98]:
tp = get_tp()

Throughput parameters:
------------------------
Fiber: 0.995
Lenses: 0.99
Grating: 0.6
Detector: 0.8
Total Throughput: 0.473


In [66]:
def get_flux_density(wavelength, mag_ab):
    """Determines the wavelength-dependent flux-density with a given AB-mag
    
    Wavelength is in nanometers.
    """

    wavelength = wavelength * u.nm

    f_ab = 3631 * u.Jy * np.power(10, (-mag_ab/2.5))
    f_lambda = f_ab.to(u.W/(u.m**2 * u.nm), equivalencies = u.spectral_density(wavelength))
    
    return f_lambda

get_flux_density(500,18)

<Quantity 2.74730542e-18 W / (nm m2)>

In [67]:
def get_sob(wavelength, mag_ab, fiber=0.995, grat=0.6, lens=0.99, detec=0.8, t_diam=40):

    t_total = get_tp(fiber, grat, lens, detec)

    f_lam = get_flux_density(wavelength, mag_ab)

    t_area = np.pi * (((t_diam * u.imperial.inch).to(u.m))/2)**2

    s_ob = f_lam * (wavelength * u.nm)/(h * c) * t_area * t_total

    return s_ob.to(1/(u.s * u.nm))

get_sob(500, 18)

Throughput parameters:
------------------------
Fiber: 0.995
Lenses: 0.99
Grating: 0.6
Detector: 0.8
Total Throughput: 0.473


<Quantity 2.65080039 1 / (nm s)>

In [78]:
rn_tot = (3.5+1.1)/2
pix_area = 6280 * 4210

rn_pp = rn_tot/pix_area
rn_pp

8.699335824621389e-08

In [99]:
def get_SNR(exp_time, wavelength, mag, ob_size=6.5, sky_noise=5.0, dark_current=0.001309, 
            read_noise=8.70e-8, fiber=0.995, grat=0.6, lens=0.99, detec=0.8, t_diam=40):
    """Predicts the SNR given an exposure time.
    
    Parameters:

    exp_time: exposure time (s)

    mag: apparent magnitude of object

    mag_zp: zero-point magnitude

    ob_size: size of object, in pixels

    sky_noise: background counts per pixel (phot/s/pix)

    dark_current: dark current (elec/s/pix)

    read_noise: time independent detector noise (elec/pix)
    """

    print('Photometric parameters:\n------------------------')
    print(f'Exposure time: {exp_time} s\nWavelength: {wavelength} nm\nAB Magnitude: {mag}\nPixel Size: {ob_size} pix^2\nSky noise: {sky_noise} phot/s/pix')
    print(f'Dark current: {dark_current} e/s/pix\nRead noise: {read_noise} e/pix\n')

    s_ob = get_sob(wavelength, mag, fiber=fiber, grat=grat, lens=lens, detec=detec, t_diam=t_diam).value

    numerator = s_ob * np.sqrt(exp_time)
    
    denominator = np.sqrt(s_ob + ob_size * (sky_noise + dark_current + (read_noise**2/exp_time)))

    snr = numerator/denominator

    print(f'\nSNR with entered parameters: {snr}')

In [100]:
get_SNR(30, 500, 18)

Photometric parameters:
------------------------
Exposure time: 30 s
Wavelength: 500 nm
AB Magnitude: 18
Pixel Size: 6.5 pix^2
Sky noise: 5.0 phot/s/pix
Dark current: 0.001309 e/s/pix
Read noise: 8.7e-08 e/pix

Throughput parameters:
------------------------
Fiber: 0.995
Lenses: 0.99
Grating: 0.6
Detector: 0.8
Total Throughput: 0.473

SNR with entered parameters: 2.4485979842212537
